In [109]:
print("Project Started")

Project Started


In [110]:
from pyspark.sql import SparkSession


In [111]:
spark = SparkSession.builder \
    .appName("Ecommerce Big Data Analytics") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [112]:
orders_df = spark.read.csv(
    "../data/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

In [113]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [114]:
from pyspark.sql.functions import col, count, when

orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [115]:
print("Total Rows:", orders_df.count())

print("Duplicate Removed Rows:",
      orders_df.dropDuplicates().count())

Total Rows: 99441
Duplicate Removed Rows: 99441


In [116]:
orders_df = orders_df.dropDuplicates()

In [117]:
from pyspark.sql.functions import to_timestamp

orders_df = orders_df.withColumn(
    "order_purchase_timestamp",
    to_timestamp("order_purchase_timestamp")
)

In [118]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [119]:
print("Total Orders:", orders_df.count())

Total Orders: 99441


In [120]:
orders_df.groupBy("order_status") \
    .count() \
    .show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
|    invoiced|  314|
|   delivered|96478|
| unavailable|  609|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [121]:
import os

os.makedirs("../outputs", exist_ok=True)